# Production-Grade MLX CNN (Apple Silicon Optimized)
Includes data loading, preprocessing, training, validation, and metrics.

In [ ]:

# Install MLX if needed
# !pip install mlx pillow scikit-learn

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split


In [ ]:

# CONFIG
IMG_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 10
DATASET_PATH = "data/fruits"  # <-- change this


In [ ]:

# LOAD DATASET
def load_images(dataset_path):
    images = []
    labels = []
    class_names = sorted(os.listdir(dataset_path))

    for label, class_name in enumerate(class_names):
        class_path = os.path.join(dataset_path, class_name)
        if not os.path.isdir(class_path):
            continue

        for file in os.listdir(class_path):
            img_path = os.path.join(class_path, file)
            try:
                img = Image.open(img_path).convert("RGB")
                img = img.resize((IMG_SIZE, IMG_SIZE))
                img = np.array(img) / 255.0
                images.append(img)
                labels.append(label)
            except:
                continue

    return np.array(images), np.array(labels), class_names

X, y, class_names = load_images(DATASET_PATH)

# Train/Val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

# Convert to MLX arrays
X_train = mx.array(X_train)
X_val = mx.array(X_val)
y_train = mx.array(y_train).reshape(-1,1)
y_val = mx.array(y_val).reshape(-1,1)


In [ ]:

# MODEL
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3), nn.ReLU(), nn.MaxPool2d(2),
        )

        self.flatten = nn.Flatten()

        self.fc = nn.Sequential(
            nn.Linear(256 * 2 * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, len(class_names))
        )

    def __call__(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        return self.fc(x)

model = CNN()


In [ ]:

# LOSS (Cross Entropy)
def loss_fn(model, X, y):
    logits = model(X)
    return mx.mean(nn.losses.cross_entropy(logits, y.squeeze().astype(mx.int32)))


In [ ]:

# ACCURACY
def accuracy(model, X, y):
    logits = model(X)
    preds = mx.argmax(logits, axis=1)
    return mx.mean((preds == y.squeeze()).astype(mx.float32))


In [ ]:

# TRAINING LOOP
optimizer = optim.Adam(learning_rate=1e-3)

for epoch in range(EPOCHS):
    loss, grads = mx.value_and_grad(model, loss_fn)(model, X_train, y_train)
    optimizer.update(model, grads)

    train_acc = accuracy(model, X_train, y_train)
    val_acc = accuracy(model, X_val, y_val)

    print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Train Acc={train_acc.item():.4f}, Val Acc={val_acc.item():.4f}")


In [ ]:

# SAVE MODEL (simple numpy export)
def save_model(model, path="model.npz"):
    params = model.parameters()
    mx.savez(path, *params)

save_model(model)
print("Model saved.")
